# Tier 2 — T2.2: Cross-dataset Validation on Pile

Re-runs Paper 2's ablation structure (144 heads + 24 layers + 30 Type A) across all 8 checkpoints of Pythia 410M, but **evaluates on Pile validation** instead of wikitext-103.

Tests the reviewer objection: "wikitext-103 specific; what about Pile validation which is closer to the training distribution?"

Central question: does the **dual differentiation signature** (count↑, eff_N↓, Gini stable) replicate?

**Compute:** ~45 min A100 (1,584 ablations). **Cost:** ~$3.

**Output:** `tier2_t22_pile.csv` + `tier2_t22_summary.json` with per-checkpoint Gini/Eff_N/count_neg and a side-by-side comparison against Paper 2's wikitext numbers.

In [ ]:
!pip install -q transformers datasets torch accelerate scipy pandas

In [ ]:
import torch, json, os, time, csv, hashlib, gc, urllib.request
import numpy as np
import pandas as pd
from scipy import stats as sp_stats
from transformers import AutoModelForCausalLM, AutoTokenizer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    print(f'GPU: {torch.cuda.get_device_name()}  |  Memory: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    OUT_DIR = '/content/drive/MyDrive/DFE_research/tier2'
    os.makedirs(OUT_DIR, exist_ok=True)
    test = os.path.join(OUT_DIR, '.w')
    with open(test, 'w') as f: f.write('ok')
    os.remove(test)
    print(f'Drive mounted and verified; output to {OUT_DIR}')
except Exception as e:
    raise RuntimeError(f'Drive mount required: {e}')

GITHUB_RAW = 'https://raw.githubusercontent.com/mool32/functional-differentiation-dfe/main'

def log(msg):
    print(f'[{time.strftime("%H:%M:%S")}] {msg}', flush=True)

## Config (matches Paper 2 exactly except eval dataset)

In [ ]:
MODEL_NAME = 'EleutherAI/pythia-410m-deduped'
N_LAYERS = 24
FIXED_HEADS = [0, 3, 6, 9, 12, 15]
CHECKPOINTS = [512, 1000, 2000, 4000, 8000, 16000, 64000, 143000]
EVAL_N_BATCHES = 25
EVAL_BATCH_SIZE = 4
EVAL_SEQ_LEN = 2048
TYPE_A_ALPHA = 1.0
N_TYPE_A = 30
SEED_BASE = 42000

CSV_PATH = os.path.join(OUT_DIR, 'tier2_t22_pile.csv')
CSV_FIELDS = ['checkpoint', 'perturbation_type', 'subtype', 'layer_idx', 'head_idx',
              'seed', 'baseline_loss', 'perturbed_loss', 'delta', 'elapsed_sec']

def csv_append(row):
    new = not os.path.exists(CSV_PATH)
    with open(CSV_PATH, 'a', newline='') as f:
        w = csv.DictWriter(f, fieldnames=CSV_FIELDS)
        if new: w.writeheader()
        w.writerow(row)

def load_completed():
    if not os.path.exists(CSV_PATH): return set()
    df = pd.read_csv(CSV_PATH)
    return set(zip(df['checkpoint'], df['perturbation_type'], df['subtype'],
                   df['layer_idx'].fillna(-1).astype(int),
                   df['head_idx'].fillna(-1).astype(int),
                   df['seed'].fillna(-1).astype(int)))

## Prepare Pile validation batches

Tries `monology/pile-uncopyrighted` (open mirror of Pile validation). Falls back to manually listed alternates.

In [ ]:
from datasets import load_dataset

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
batches_path = os.path.join(OUT_DIR, 'pile_val_batches.pt')

PILE_SPECS = [
    ('monology/pile-uncopyrighted', None, 'validation'),
    ('NeelNanda/pile-10k', None, 'train'),
    ('mit-han-lab/pile-val-backup', None, 'validation'),
]

if os.path.exists(batches_path):
    batches = torch.load(batches_path, weights_only=True)
    log(f'Cached Pile batches: {batches.shape}')
else:
    batches = None
    for spec in PILE_SPECS:
        try:
            log(f'Trying {spec[0]}...')
            ds = load_dataset(spec[0], spec[1], split=spec[2], streaming=True)
            need = EVAL_N_BATCHES * EVAL_BATCH_SIZE * EVAL_SEQ_LEN
            toks = []
            for ex in ds:
                text = ex.get('text', ex.get('content', ''))
                if not text or len(text.strip()) < 50: continue
                ids = tok(text, return_tensors='pt', truncation=False)['input_ids'].squeeze()
                if ids.dim() == 0: continue
                toks.append(ids)
                if sum(t.numel() for t in toks) >= need * 1.2: break
            if sum(t.numel() for t in toks) >= need:
                batches = torch.cat(toks)[:need].reshape(EVAL_N_BATCHES, EVAL_BATCH_SIZE, EVAL_SEQ_LEN)
                torch.save(batches, batches_path)
                log(f'OK: {batches.shape} from {spec[0]}')
                break
        except Exception as e:
            log(f'  {spec[0]} failed: {e}')
    if batches is None:
        raise RuntimeError('Could not load any Pile variant. Check HF access.')

## Ablation primitives

In [ ]:
HEAD_DIM = None  # set after model load

def ablate_head(model, L, H):
    hd = model.config.hidden_size // model.config.num_attention_heads
    w = model.gpt_neox.layers[L].attention.dense.weight
    saved = w.data.clone()
    w.data[:, H*hd:(H+1)*hd] = 0
    return saved

def restore_head(model, L, saved):
    model.gpt_neox.layers[L].attention.dense.weight.data.copy_(saved)

def ablate_layer(model, L):
    block = model.gpt_neox.layers[L]
    saved = {n: p.data.clone() for n, p in block.named_parameters()}
    for _, p in block.named_parameters():
        p.data.zero_()
    return saved

def restore_block(model, L, saved):
    block = model.gpt_neox.layers[L]
    for n, p in block.named_parameters():
        p.data.copy_(saved[n])

def perturb_type_a(model, L, alpha, seed):
    g = torch.Generator(device=device).manual_seed(seed)
    block = model.gpt_neox.layers[L]
    saved = {n: p.data.clone() for n, p in block.named_parameters()}
    for _, p in block.named_parameters():
        noise = torch.randn(p.shape, generator=g, device=device, dtype=p.dtype) * (p.data.std() * alpha)
        p.data.add_(noise)
    return saved

@torch.no_grad()
def evaluate_loss(model):
    total = 0.0
    for i in range(batches.shape[0]):
        ids = batches[i].to(device)
        total += model(input_ids=ids, labels=ids).loss.item()
    return total / batches.shape[0]

## Main sweep

In [ ]:
completed = load_completed()
log(f'{len(completed)} rows already done, will skip')

for step in CHECKPOINTS:
    log(f'=== step{step} ===')
    t_ck = time.time()
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, revision=f'step{step}', torch_dtype=torch.float32
    ).to(device).eval()
    bl = evaluate_loss(model)
    log(f'  baseline loss (Pile val): {bl:.4f}')

    # 144 heads
    for L in range(N_LAYERS):
        for H in FIXED_HEADS:
            key = (step, 'head', 'output_zeroing', L, H, -1)
            if key in completed: continue
            t0 = time.time()
            saved = ablate_head(model, L, H)
            pl = evaluate_loss(model)
            restore_head(model, L, saved)
            csv_append({
                'checkpoint': step, 'perturbation_type': 'head', 'subtype': 'output_zeroing',
                'layer_idx': L, 'head_idx': H, 'seed': -1,
                'baseline_loss': bl, 'perturbed_loss': pl,
                'delta': -(pl - bl) / abs(bl), 'elapsed_sec': time.time() - t0,
            })

    # 24 layers
    for L in range(N_LAYERS):
        key = (step, 'layer', 'full_zero', L, -1, -1)
        if key in completed: continue
        t0 = time.time()
        saved = ablate_layer(model, L)
        pl = evaluate_loss(model)
        restore_block(model, L, saved)
        csv_append({
            'checkpoint': step, 'perturbation_type': 'layer', 'subtype': 'full_zero',
            'layer_idx': L, 'head_idx': -1, 'seed': -1,
            'baseline_loss': bl, 'perturbed_loss': pl,
            'delta': -(pl - bl) / abs(bl), 'elapsed_sec': time.time() - t0,
        })

    # 30 Type A
    for i in range(N_TYPE_A):
        L = i % N_LAYERS
        seed = SEED_BASE + i
        key = (step, 'type_a', 'block_noise', L, -1, seed)
        if key in completed: continue
        t0 = time.time()
        saved = perturb_type_a(model, L, TYPE_A_ALPHA, seed)
        pl = evaluate_loss(model)
        restore_block(model, L, saved)
        csv_append({
            'checkpoint': step, 'perturbation_type': 'type_a', 'subtype': 'block_noise',
            'layer_idx': L, 'head_idx': -1, 'seed': seed,
            'baseline_loss': bl, 'perturbed_loss': pl,
            'delta': -(pl - bl) / abs(bl), 'elapsed_sec': time.time() - t0,
        })

    log(f'  checkpoint done in {(time.time() - t_ck)/60:.1f} min')
    del model; gc.collect(); torch.cuda.empty_cache()

log('sweep complete')

## Analysis — compare Pile vs wikitext (Paper 2)

In [ ]:
# Pull Paper 2 wikitext for side-by-side
p2_path = os.path.join(OUT_DIR, 'paper2_all_ablations.csv')
if not os.path.exists(p2_path):
    urllib.request.urlretrieve(f'{GITHUB_RAW}/data/all_ablations.csv', p2_path)
paper2 = pd.read_csv(p2_path)

pile = pd.read_csv(CSV_PATH)

def metrics(df):
    heads = df[df['perturbation_type'] == 'head']['delta'].values
    neg = -heads[heads < -1e-4]
    if len(neg) < 5:
        return {'count_neg': 0, 'eff_n': float('nan'), 'gini': float('nan')}
    s = np.sort(neg)
    n = len(s)
    gini = (2 * np.sum((np.arange(1, n+1)) * s)) / (n * s.sum()) - (n + 1) / n
    eff_n = s.sum() ** 2 / (s ** 2).sum()
    return {'count_neg': int(len(neg)), 'eff_n': float(eff_n), 'gini': float(gini)}

comparison = {}
print(f'{"step":>6} | {"WT count":>8} {"Pile count":>10} | {"WT eff_n":>8} {"Pile eff_n":>10} | {"WT gini":>8} {"Pile gini":>8}')
print('-' * 85)
for step in CHECKPOINTS:
    wt = metrics(paper2[paper2['checkpoint'] == step])
    pl = metrics(pile[pile['checkpoint'] == step])
    comparison[step] = {'wikitext': wt, 'pile': pl}
    print(f'{step:>6} | {wt["count_neg"]:>8} {pl["count_neg"]:>10} | '
          f'{wt["eff_n"]:>8.2f} {pl["eff_n"]:>10.2f} | '
          f'{wt["gini"]:>8.3f} {pl["gini"]:>8.3f}')

# Dual differentiation verdict for Pile
first, last = CHECKPOINTS[0], CHECKPOINTS[-1]
pf, pl = comparison[first]['pile'], comparison[last]['pile']
dd_signature = (pl['count_neg'] > pf['count_neg']
                and pl['eff_n'] < pf['eff_n']
                and abs(pl['gini'] - pf['gini']) < 0.1)
print(f'\nPile dual-differentiation signature: {dd_signature}')

summary = {
    'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
    'comparison_per_checkpoint': comparison,
    'pile_dual_differentiation_signature': bool(dd_signature),
}
summary_path = os.path.join(OUT_DIR, 'tier2_t22_summary.json')
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2, default=str)
log(f'saved {summary_path}')

try:
    from google.colab import files
    files.download(CSV_PATH)
    files.download(summary_path)
except Exception:
    pass